# mssql_python — praktyczny przewodnik: połączenie, odczyt do DataFrame, operacje DML

Notebook krok po kroku dla oficjalnego, nowego (GA) sterownika Microsoftu
`mssql-python` (pakiet PyPI: `mssql-python`, moduł do importu: `mssql_python`) —
następcy `pyodbc` w wielu nowych projektach, bo nie wymaga osobnej instalacji
sterownika ODBC (bundluje DDBC — Direct Database Connectivity).

Kod jest pisany **bezpośrednio** tam, gdzie logika jest prosta — łatwiej wtedy
widzieć, co się dzieje. W funkcje owinięte są tylko fragmenty, które
realistycznie będziesz kopiować między projektami (insert partiami, insert
tylko nowych rekordów) — to samo podejście co w poprzednim notebooku o Folium.

⚠️ **Komórki łączące się z bazą wymagają Twojego connection stringa** i nie
zostały wykonane w tym środowisku (brak dostępu do zewnętrznego SQL Servera).
Podmień `CONNECTION_STRING` na własne dane i uruchom lokalnie — składnia jest
zweryfikowana z oficjalną dokumentacją Microsoftu (stan na 2026, wersja GA).

**Instalacja:**
```
pip install mssql-python pandas polars duckdb pyarrow python-dotenv
```

In [ ]:
from pathlib import Path
import re

import pandas as pd
import polars as pl
import duckdb
from mssql_python import connect

## 1. Połączenie z bazą danych

### Connection string — co jest wymagane, a co nie

| Element | Wymagany? | Komentarz |
|---|---|---|
| `Server` | **tak** | nazwa serwera, opcjonalnie `,port` (domyślnie 1433) |
| `Database` | **tak** (w praktyce) | bez tego łączysz się do bazy domyślnej użytkownika |
| `UID` / `PWD` | tak — **tylko przy autentykacji SQL** | pomiń przy `Trusted_Connection=yes` (Windows) lub `Authentication=ActiveDirectory...` (Azure/Entra) |
| `Encrypt` | nie, ale zalecane `yes` | wymagane domyślnie w nowszych wersjach sterownika |
| `TrustServerCertificate` | nie | ustaw `yes` tylko lokalnie z certyfikatem self-signed; w produkcji zostaw `no` |
| `Trusted_Connection` | nie | `yes` = użyj zalogowanego konta Windows zamiast UID/PWD |
| `Authentication` | nie | np. `ActiveDirectoryDefault`/`ActiveDirectoryInteractive` — do Azure SQL bez hasła w kodzie |

Connection string trzymaj w zmiennych środowiskowych / pliku `.env`
(`python-dotenv`) — nigdy na sztywno w repo.

In [ ]:
# --- SQL Server lokalny / on-premise, autentykacja SQL (typowy przypadek w pracy) ---
CONNECTION_STRING = (
    "Server=localhost,1433;"
    "Database=MojaBaza;"
    "UID=uzytkownik;"
    "PWD=haslo;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes"   # tylko na potrzeby lokalnego dev
)

# --- Azure SQL / Fabric, bez hasła w kodzie (Entra ID) ---
# CONNECTION_STRING = (
#     "Server=twoj_serwer.database.windows.net;"
#     "Database=MojaBaza;"
#     "Encrypt=yes;"
#     "Authentication=ActiveDirectoryDefault"
# )

# --- z .env zamiast wpisywania na sztywno ---
# from dotenv import load_dotenv
# from os import getenv
# load_dotenv()
# CONNECTION_STRING = getenv("SQL_CONNECTION_STRING")

conn = connect(CONNECTION_STRING)
cursor = conn.cursor()

### Ważne właściwości sterownika (inne niż w `pyodbc`)

- **Domyślnie `autocommit=False`** — każda instrukcja żyje w transakcji, dopóki
  nie wywołasz `conn.commit()` (albo `conn.rollback()`). To dobra wiadomość
  dla operacji DML z sekcji 3 — TRUNCATE + INSERT łatwo zamknąć w jedną
  atomową transakcję.
- **Brak MARS** (Multiple Active Result Sets) — na jednym połączeniu może być
  aktywny tylko jeden "niewyczerpany" wynik zapytania na raz. Zanim odpalisz
  kolejne `execute()` na tym samym cursorze (albo drugim cursorze tego
  samego connection), dociągnij wszystkie wiersze poprzedniego zapytania
  (`fetchall()`), albo użyj osobnego połączenia.
- **Context manager** — `with connect(...) as conn:` domyka połączenie na
  wyjściu z bloku (ale NIE robi automatycznego commit/rollback za Ciebie
  poza przypadkiem wyjątku — i tak jawne `conn.commit()` jest najbezpieczniejsze).

In [ ]:
# Bezpieczny wzorzec otwierania/zamykania połączenia — poleca się w skryptach/pipeline'ach
with connect(CONNECTION_STRING) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT 1 AS test")
    print(cursor.fetchone())
    conn.commit()
# połączenie zamknięte automatycznie na wyjściu z bloku `with`

## 2. Pobranie tabeli / wyniku zapytania SQL do DataFrame

### a) Najprościej — `pandas.read_sql` / `polars.read_database`

Obie biblioteki potrafią wykonać zapytanie bezpośrednio na dowolnym
połączeniu zgodnym z DB-API 2.0 (a `mssql_python` takim jest) — nie trzeba
ręcznie obsługiwać cursora.

In [ ]:
sql = "SELECT TOP 100 * FROM Sales.Customers"

df_pandas = pd.read_sql(sql, conn)      # najprostszy sposób w Pandas
df_polars = pl.read_database(sql, conn) # analogicznie w Polars

df_pandas.head()

> Uwaga: `pandas.read_sql` może wypisać `UserWarning` o "tylko SQLAlchemy
> connectable są w pełni wspierane" — to nieszkodliwe; `mssql_python`
> implementuje standardowe DB-API 2.0, więc odczyt działa poprawnie.
> `polars.read_database` z surowym połączeniem DB-API bywa wolniejszy niż
> `read_database_uri` (który korzysta z `connectorx`/`adbc` i pomija Pythona
> przy konwersji do Arrow) — przy naprawdę dużych zapytaniach warto to
> rozważyć, o ile masz gotowy connection URI zamiast connection stringa ODBC.

### b) Ręcznie przez cursor — kiedy potrzebujesz pełnej kontroli

Przydatne, gdy chcesz np. samemu decydować o typach kolumn, przetwarzać
wynik strumieniowo (`fetchmany`) zamiast ładować wszystko naraz, albo gdy
`read_sql`/`read_database` z jakiegoś powodu nie radzi sobie z konkretnym
typem danych.

In [ ]:
cursor = conn.cursor()
cursor.execute(sql)

kolumny = [opis[0] for opis in cursor.description]   # cursor.description = (nazwa, typ, ...) per kolumna
wiersze = cursor.fetchall()                            # lista obiektów Row

df_manual = pd.DataFrame((tuple(w) for w in wiersze), columns=kolumny)
# odpowiednik w Polars:
df_manual_pl = pl.DataFrame((tuple(w) for w in wiersze), schema=kolumny, orient="row")

### c) Odczyt dużego wyniku partiami (`fetchmany`) zamiast `fetchall`

Przy zapytaniach zwracających miliony wierszy `fetchall()` ładuje wszystko
naraz do pamięci. `fetchmany()` (rozmiar paczki: `cursor.arraysize`) pozwala
przetwarzać wynik strumieniowo — np. dopisując kolejne paczki bezpośrednio
do pliku Parquet zamiast trzymać cały DataFrame w RAM-ie.

In [ ]:
cursor.execute("SELECT * FROM Sales.OrderDetails")
cursor.arraysize = 5000   # rozmiar jednej paczki

wszystkie_ramki = []
while True:
    paczka = cursor.fetchmany()
    if not paczka:
        break
    kolumny = [o[0] for o in cursor.description]
    wszystkie_ramki.append(pl.DataFrame((tuple(w) for w in paczka), schema=kolumny, orient="row"))

df_streamed = pl.concat(wszystkie_ramki) if wszystkie_ramki else pl.DataFrame()

## 3. Pobranie danych na podstawie zapytania zapisanego w pliku `.sql`

Trzymanie zapytań w osobnych plikach `.sql` (zamiast stringów w kodzie
Pythona) daje podświetlanie składni w edytorze, łatwiejsze code review i
możliwość współdzielenia zapytań między Pythonem a np. DAX Studio/SSMS.

In [ ]:
sciezka_sql = Path("queries/klienci_aktywni.sql")

# treść pliku, np.:
# SELECT KlientID, Nazwa, Miasto, DataAktywacji
# FROM Sales.Customers
# WHERE Aktywny = 1

sql_z_pliku = sciezka_sql.read_text(encoding="utf-8")
df = pd.read_sql(sql_z_pliku, conn)

Jeśli zapytanie w pliku ma placeholdery, wykonuj je jako zapytanie
parametryzowane (`cursor.execute(sql, params)`), **nie** przez `.format()`/
f-string wklejający wartości do tekstu SQL — to otwiera na SQL injection,
nawet gdy zapytanie pochodzi "tylko" z Twojego własnego pliku (parametry
mogą pochodzić od użytkownika/formularza wyżej w pipeline).

In [ ]:
# plik queries/klienci_wg_miasta.sql zawiera:
# SELECT KlientID, Nazwa FROM Sales.Customers WHERE Miasto = %(miasto)s

sql_z_parametrem = Path("queries/klienci_wg_miasta.sql").read_text(encoding="utf-8")

cursor.execute(sql_z_parametrem, {"miasto": "Warszawa"})
df_warszawa = pd.DataFrame(
    (tuple(w) for w in cursor.fetchall()),
    columns=[o[0] for o in cursor.description],
)

## 4. Operacje DML — zapis DataFrame do SQL Server

Wspólny punkt wyjścia dla wszystkich przykładów poniżej: DataFrame gotowy do
zapisu (np. po transformacjach w Polars/DuckDB) plus gotowa instrukcja
`INSERT` z placeholderami `%(nazwa_kolumny)s` (domyślny styl `pyformat`
sterownika — nazwy placeholderów muszą odpowiadać kluczom w słowniku).

In [ ]:
dane = pl.DataFrame({
    "KlientID": [101, 102, 103],
    "Nazwa": ["Firma A", "Firma B", "Firma C"],
    "Miasto": ["Warszawa", "Kraków", "Gdańsk"],
})

insert_sql = """
INSERT INTO dbo.Klienci (KlientID, Nazwa, Miasto)
VALUES (%(KlientID)s, %(Nazwa)s, %(Miasto)s)
"""

rekordy = dane.to_dicts()          # Polars -> lista dictów
# rekordy = dane.to_pandas().to_dict("records")   # gdybyś startował z Pandas
rekordy[:2]

### a) Insert DataFrame do istniejącej tabeli

`executemany()` używa **wiązania parametrów kolumnami** (column-wise) — to
znacznie szybsze niż wywoływanie `execute()` w pętli po pojedynczych
wierszach, bo sterownik wysyła dane do serwera w większych porcjach zamiast
jednego round-tripu na wiersz.

In [ ]:
cursor.executemany(insert_sql, rekordy)
conn.commit()

print(f"Wstawiono {cursor.rowcount} wierszy")

### b) TRUNCATE + INSERT (pełne przeładowanie tabeli)

Klasyczny wzorzec "full load" w ETL/ELT — czyścisz tabelę docelową i
ładujesz ją od nowa. `TRUNCATE` jest szybszy niż `DELETE` (nie loguje
usunięcia wiersz po wierszu) i resetuje `IDENTITY`, ale **wymaga uprawnienia
`ALTER` na tabeli i nie zadziała, jeśli inna tabela ma klucz obcy wskazujący
na tę tabelę** — w takim wypadku użyj `DELETE FROM ...` zamiast `TRUNCATE`.

Obie instrukcje muszą być w jednej transakcji: jeśli TRUNCATE się powiedzie,
a INSERT nie, `rollback()` musi cofnąć też TRUNCATE — inaczej zostajesz z
pustą tabelą.

In [ ]:
try:
    cursor.execute("TRUNCATE TABLE dbo.Klienci")
    cursor.executemany(insert_sql, rekordy)
    conn.commit()
    print(f"Przeładowano tabelę — {cursor.rowcount} wierszy")
except Exception:
    conn.rollback()
    raise

### c) Insert tylko rekordów, które jeszcze nie istnieją w tabeli

Dwa podejścia — wybór zależy od wolumenu danych.

**Po stronie Pythona (anti-join w Polars/Pandas)** — proste i wystarczające,
gdy zbiór kluczy istniejących w tabeli mieści się wygodnie w pamięci
(rzędu pojedynczych milionów wierszy):

In [ ]:
istniejace = pl.read_database("SELECT KlientID FROM dbo.Klienci", conn)["KlientID"].to_list()

nowe_rekordy = dane.filter(~pl.col("KlientID").is_in(istniejace))

if nowe_rekordy.height > 0:
    cursor.executemany(insert_sql, nowe_rekordy.to_dicts())
    conn.commit()

print(f"Dodano {nowe_rekordy.height} nowych rekordów (pominięto {dane.height - nowe_rekordy.height} już istniejących)")

**Po stronie SQL (tabela tymczasowa + `NOT EXISTS`)** — lepsze przy dużych
wolumenach, bo porównanie kluczy robi silnik SQL na serwerze (z indeksem),
zamiast ściągać wszystkie klucze do Pythona. To też baza pod pełny **upsert**
(`MERGE`), gdyby kiedyś potrzebne było też aktualizowanie istniejących
rekordów, a nie tylko dokładanie nowych.

In [ ]:
cursor.execute("""
IF OBJECT_ID('tempdb..#StagingKlienci') IS NOT NULL DROP TABLE #StagingKlienci;
CREATE TABLE #StagingKlienci (KlientID INT, Nazwa NVARCHAR(200), Miasto NVARCHAR(100));
""")

cursor.executemany(
    "INSERT INTO #StagingKlienci (KlientID, Nazwa, Miasto) VALUES (%(KlientID)s, %(Nazwa)s, %(Miasto)s)",
    dane.to_dicts(),
)

cursor.execute("""
INSERT INTO dbo.Klienci (KlientID, Nazwa, Miasto)
SELECT s.KlientID, s.Nazwa, s.Miasto
FROM #StagingKlienci s
WHERE NOT EXISTS (
    SELECT 1 FROM dbo.Klienci k WHERE k.KlientID = s.KlientID
);
""")

conn.commit()
print(f"Dodano {cursor.rowcount} nowych rekordów (INSERT ... WHERE NOT EXISTS)")

> Wariant z pełnym upsertem (`MERGE`), gdyby był potrzebny — po stronie SQL,
> na tej samej tabeli tymczasowej:
> ```sql
> MERGE dbo.Klienci AS target
> USING #StagingKlienci AS source
>     ON target.KlientID = source.KlientID
> WHEN MATCHED THEN
>     UPDATE SET target.Nazwa = source.Nazwa, target.Miasto = source.Miasto
> WHEN NOT MATCHED THEN
>     INSERT (KlientID, Nazwa, Miasto) VALUES (source.KlientID, source.Nazwa, source.Miasto);
> ```

### d) Insert partiami — gdy DataFrame jest duży (10 tys.+ rekordów)

To jedna z niewielu rzeczy w tym notebooku, którą warto mieć jako funkcję —
używasz jej w praktycznie każdym pipeline'ie ładującym więcej niż kilka
tysięcy wierszy, więc pisanie jej na nowo za każdym razem nie ma sensu.

Chunking ma dwa uzasadnienia praktyczne:
1. **Kontrola rozmiaru transakcji** — jeden ogromny `executemany()` na 500
   tys. wierszy trzyma otwartą transakcję bardzo długo (blokady, log
   transakcyjny rośnie); łatwiej commitować co N wierszy.
2. **Widoczność postępu i punkt przywracania** — przy błędzie w połowie
   wiesz, ile już się załadowało, zamiast zaczynać całość od zera.

In [ ]:
def insert_in_batches(cursor, sql: str, records: list[dict], batch_size: int = 2000, commit_every_batch: bool = True) -> int:
    """
    Wstawia listę rekordów (dictów) do SQL Server w paczkach po `batch_size`.

    commit_every_batch=True  -> commit po każdej paczce (bezpieczniejsze przy
        bardzo dużych wolumenach — błąd nie cofa już zapisanych paczek).
    commit_every_batch=False -> jeden commit na końcu (pełna atomowość
        "wszystko albo nic", kosztem dłużej otwartej transakcji).

    Zwraca łączną liczbę wstawionych wierszy.
    """
    total = len(records)
    wstawiono = 0

    for start in range(0, total, batch_size):
        batch = records[start:start + batch_size]
        cursor.executemany(sql, batch)
        wstawiono += len(batch)

        if commit_every_batch:
            cursor.connection.commit()

        print(f"  {wstawiono}/{total} wierszy wstawionych")

    if not commit_every_batch:
        cursor.connection.commit()

    return wstawiono


# przykład na większym, syntetycznym zbiorze
duzy_df = pl.DataFrame({
    "KlientID": list(range(1000, 11000)),
    "Nazwa": [f"Firma {i}" for i in range(10000)],
    "Miasto": ["Warszawa"] * 10000,
})

insert_in_batches(cursor, insert_sql, duzy_df.to_dicts(), batch_size=2000)

> Uwaga: `cursor.connection` — jeśli Twoja wersja sterownika nie eksponuje
> tego atrybutu, przekaż `conn` do funkcji jako osobny parametr
> (`insert_in_batches(cursor, conn, sql, records, ...)`) i wywołuj
> `conn.commit()` bezpośrednio — sprawdź szybko w swoim środowisku
> (`hasattr(cursor, "connection")`), obie wersje sterownika bywały w obiegu.

## 5. Inne przydatne przypadki użycia (analiza + pipeline'y)

### a) UPDATE na podstawie DataFrame (ten sam wzorzec co INSERT)

In [ ]:
zmiany = pl.DataFrame({
    "KlientID": [101, 102],
    "Nazwa": ["Firma A (zmieniona nazwa)", "Firma B Sp. z o.o."],
    "Miasto": ["Warszawa", "Kraków"],
})

update_sql = """
UPDATE dbo.Klienci
SET Nazwa = %(Nazwa)s, Miasto = %(Miasto)s
WHERE KlientID = %(KlientID)s
"""

cursor.executemany(update_sql, zmiany.to_dicts())
conn.commit()
print(f"Zaktualizowano {cursor.rowcount} wierszy")

### b) DuckDB jako silnik transformacji przed zapisem do SQL Server

DuckDB potrafi odpytywać DataFrame Polars/Pandas bezpośrednio po nazwie
zmiennej (bez kopiowania danych) — wygodne do szybkich agregacji/joinów
tuż przed wysłaniem wyniku z powrotem do SQL Server, szczególnie gdy
transformacja łączy dane z kilku różnych źródeł (np. wynik zapytania SQL +
plik CSV + inny DataFrame).

In [ ]:
podsumowanie = duckdb.sql("""
    SELECT Miasto, COUNT(*) AS liczba_klientow
    FROM dane
    GROUP BY Miasto
    ORDER BY liczba_klientow DESC
""").pl()   # .pl() -> Polars, .df() -> Pandas, .arrow() -> Arrow Table

podsumowanie

### c) Watermark / ładowanie przyrostowe (incremental load)

Zamiast ściągać całą tabelę źródłową za każdym razem — typowy wzorzec w
pipeline'ach (Airflow, harmonogramy nocne): pobierz tylko rekordy zmienione
od ostatniego uruchomienia, zapisując "znak wodny" (np. max. `DataModyfikacji`
poprzedniego przebiegu) w osobnej tabeli kontrolnej.

In [ ]:
ostatni_watermark = cursor.execute(
    "SELECT MAX(WartoscWatermark) FROM etl.Watermarks WHERE NazwaProcesu = %(nazwa)s",
    {"nazwa": "load_klienci"},
).fetchval()

sql_przyrostowy = """
SELECT KlientID, Nazwa, Miasto, DataModyfikacji
FROM Sales.Customers
WHERE DataModyfikacji > %(watermark)s
"""
cursor.execute(sql_przyrostowy, {"watermark": ostatni_watermark})
df_przyrost = pd.DataFrame((tuple(w) for w in cursor.fetchall()), columns=[o[0] for o in cursor.description])

# ... insert_in_batches(...) do tabeli docelowej, a na końcu:
nowy_watermark = df_przyrost["DataModyfikacji"].max()
cursor.execute(
    "UPDATE etl.Watermarks SET WartoscWatermark = %(w)s WHERE NazwaProcesu = %(nazwa)s",
    {"w": nowy_watermark, "nazwa": "load_klienci"},
)
conn.commit()

### d) Wykonanie skryptu `.sql` z wieloma instrukcjami (separator `GO`)

`GO` to polecenie klienta SSMS/sqlcmd, a nie T-SQL — sterowniki takie jak
`mssql_python` go nie rozumieją i wykonanie całego skryptu z `GO` jako
jednego `execute()` zwróci błąd składni. Skrypt trzeba samodzielnie podzielić
na batch'e po liniach zawierających tylko `GO`.

In [ ]:
skrypt = Path("migracja.sql").read_text(encoding="utf-8")

batche = [b.strip() for b in re.split(r"(?im)^\s*GO\s*$", skrypt) if b.strip()]

for batch in batche:
    cursor.execute(batch)
conn.commit()

print(f"Wykonano {len(batche)} batchy skryptu")

### e) Szybkie sprawdzenie pojedynczej wartości — `fetchval()`

Wygodne do kontroli w pipeline (liczba wierszy przed/po, suma kontrolna),
bez rozpakowywania `fetchone()[0]`.

In [ ]:
liczba_wierszy = cursor.execute("SELECT COUNT(*) FROM dbo.Klienci").fetchval()
print(f"Tabela dbo.Klienci: {liczba_wierszy} wierszy")

### f) Bezpieczne zamykanie zasobów na końcu skryptu/pipeline'u

In [ ]:
cursor.close()
conn.close()

## Podsumowanie

- **Odczyt:** `pd.read_sql(sql, conn)` / `pl.read_database(sql, conn)` do
  większości przypadków; ręczny `cursor` + `fetchmany()` tylko gdy zapytanie
  zwraca naprawdę dużo danych albo potrzebujesz kontroli nad typami.
- **Zapis:** zawsze `executemany()` z listą dictów (`df.to_dicts()` /
  `df.to_dict("records")"), nigdy pętla z pojedynczymi `execute()` per wiersz.
- **Duże wolumeny (10k+):** `insert_in_batches()` + rozważ porównanie kluczy
  po stronie SQL (staging + `NOT EXISTS`/`MERGE`), a nie w Pythonie, jeśli
  tabela docelowa jest naprawdę duża.
- **Transformacje przed zapisem:** Polars/DuckDB robią cięższą robotę
  (agregacje, joiny, czyszczenie) w pamięci — `mssql_python` służy tylko do
  "ostatniej mili" (odczyt źródła, zapis wyniku), zgodnie z tym samym
  podziałem odpowiedzialności, którego pilnujesz w Power BI (SQL/Python do
  ciężkich transformacji, nie DAX).
- To wszystko naturalnie mapuje się na taski Airflow: `extract` (sekcja 2/3)
  → `transform` (Polars/DuckDB) → `load` (sekcja 4), każdy jako osobny task
  w DAG-u.